In [1]:
!pip install kagglehub pandas

In [3]:
import os
import numpy as np
import kagglehub
import pandas as pd

# On pose les paramètres principaux
OUTPUT_CSV = "perfumes_200_clean.csv"
MIN_VOTES = 500
MIN_PEOPLE = 50
MIN_DESC_LEN = 250
PER_FAMILY = 25
GENDER_QUOTAS = {"female": 8, "male": 8, "unisex": 9}

BLACKLIST_BRANDS = {
    "spongebob squarepants", "shrek", "minions", "barbie", "disney",
    "hello kitty", "marvel", "star wars", "pokemon", "minecraft"
}

FAMILY_MAP = {
    "woody": "Woody", "oud": "Woody", "earthy": "Woody", "mossy": "Woody", "conifer": "Woody",
    "floral": "Floral", "white floral": "Floral", "yellow floral": "Floral",
    "rose": "Floral", "iris": "Floral", "violet": "Floral", "tuberose": "Floral",
    "citrus": "Citrus & Fresh", "fresh": "Citrus & Fresh", "aquatic": "Citrus & Fresh",
    "marine": "Citrus & Fresh", "ozonic": "Citrus & Fresh", "green": "Citrus & Fresh",
    "aromatic": "Aromatic", "herbal": "Aromatic", "lavender": "Aromatic", "fresh spicy": "Aromatic",
    "amber": "Amber & Spicy", "warm spicy": "Amber & Spicy", "balsamic": "Amber & Spicy",
    "smoky": "Amber & Spicy", "incense": "Amber & Spicy", "resinous": "Amber & Spicy",
    "sweet": "Sweet & Gourmand", "vanilla": "Sweet & Gourmand", "caramel": "Sweet & Gourmand",
    "honey": "Sweet & Gourmand", "cacao": "Sweet & Gourmand", "coffee": "Sweet & Gourmand",
    "almond": "Sweet & Gourmand", "lactonic": "Sweet & Gourmand",
    "fruity": "Fruity", "tropical": "Fruity", "cherry": "Fruity", "coconut": "Fruity",
    "leather": "Leather & Powdery", "musky": "Leather & Powdery",
    "powdery": "Leather & Powdery", "animalic": "Leather & Powdery", "soft spicy": "Leather & Powdery",
}

ACCESSIBLE_BRANDS = {
    "lattafa perfumes", "armaf", "afnan", "al haramain perfumes", "rasasi",
    "zara", "nautica", "davidoff", "calvin klein", "elizabeth arden",
    "bath & body works", "victoria's secret", "ariana grande", "britney spears",
    "lalique", "rochas", "bentley", "montblanc", "coach", "guess", "nusuk",
    "dumont", "natura", "o boticário", "maison alhambra", "french avenue",
    "paris corner", "zimaya", "antonio banderas", "tommy hilfiger", "mercedes-benz"
}

LUXURY_NICHE_BRANDS = {
    "creed", "tom ford", "parfums de marly", "xerjoff", "maison francis kurkdjian",
    "byredo", "le labo", "diptyque", "jo malone london", "louis vuitton",
    "amouage", "initio parfums prives", "by kilian", "kilian", "roja dove",
    "nishane", "penhaligon's", "frederic malle", "memo paris", "ex nihilo",
    "clive christian", "bond no 9", "mancera", "montale", "sospiro perfumes",
    "ormonde jayne", "acqua di palma", "acqua di parma", "maison crivelli",
    "bdk parfums", "nasomatto", "orto parisi", "tiziana terenzi",
    "marc-antoine barrois", "the spirit of dubai", "arabian oud", "loumari",
    "serge lutens", "profumum roma", "kajal", "electimuss", "zoologist perfumes"
}

SEASON_ORDER = ["Spring", "Summer", "Autumn", "Winter"]

# On charge le dataset brut et ensuite on filtre seulement ce qu'on veut garder
path = kagglehub.dataset_download("ledecanteur/fragrantica-perfumes")
use_cols = [
    "name", "brand", "gender", "rating_avg", "vote_count",
    "longevity_avg", "sillage_avg", "price_value_avg",
    "winter", "spring", "summer", "autumn", "day", "night", "people",
    "accords", "notes_top", "notes_middle", "notes_base", "notes_flat", "description"
]
df = pd.read_csv(os.path.join(path, "perfumes.csv"), usecols=use_cols)

df = df.dropna(subset=[
    "name", "brand", "gender", "description", "accords",
    "rating_avg", "longevity_avg", "sillage_avg"
])

brand_low = df["brand"].str.strip().str.lower()
has_notes = df[["notes_top", "notes_middle", "notes_base", "notes_flat"]].notna().any(axis=1)

df = df[
    (~brand_low.isin(BLACKLIST_BRANDS)) &
    (df["vote_count"] >= MIN_VOTES) &
    (df["people"] >= MIN_PEOPLE) &
    (df["description"].str.len() >= MIN_DESC_LEN) &
    has_notes
].copy()

# On crée nos familles, le score de qualité, l'équilibrage famille x genre
df["Family"] = (
    df["accords"]
    .str.split("|").str[0]
    .str.rsplit(":", n=1).str[0]
    .str.strip().str.lower()
    .map(FAMILY_MAP)
)
df["gender"] = df["gender"].str.strip().str.lower()
df = df.dropna(subset=["Family"]).drop_duplicates(subset=["name", "brand"])

df["QualityScore"] = df["rating_avg"] * np.log1p(df["vote_count"])
df = df.sort_values("QualityScore", ascending=False)

selected = []
for _, group in df.groupby("Family"):
    picks = [group[group["gender"] == g].head(q) for g, q in GENDER_QUOTAS.items()]
    fam_df = pd.concat(picks)
    if len(fam_df) < PER_FAMILY:
        rest = group.drop(index=fam_df.index).head(PER_FAMILY - len(fam_df))
        fam_df = pd.concat([fam_df, rest])
    selected.append(fam_df)

df_bal = pd.concat(selected).reset_index(drop=True)

# Nettoyage et création des colonnes
def clean_pipe(val, max_items=5):
    if pd.isna(val) or not str(val).strip():
        return ""
    return ", ".join(item.rsplit(":", 1)[0].strip() for item in str(val).split("|") if item.strip())[:None]

def clean_pipe_list(val, max_items=5):
    if pd.isna(val) or not str(val).strip():
        return ""
    items = [item.rsplit(":", 1)[0].strip() for item in str(val).split("|") if item.strip()]
    return ", ".join(items[:max_items])

# Notes séparées + combinées directement (sans regex ni relecture CSV)
top_notes = df_bal["notes_top"].apply(lambda x: clean_pipe_list(x, 5))
heart_notes = df_bal["notes_middle"].apply(lambda x: clean_pipe_list(x, 5))
base_notes = df_bal["notes_base"].apply(lambda x: clean_pipe_list(x, 5))
flat_notes = df_bal["notes_flat"].apply(lambda x: clean_pipe_list(x, 8))

is_flat = (top_notes == "") & (heart_notes == "") & (base_notes == "")
top_final = np.where(is_flat, flat_notes, top_notes)
heart_final = np.where(is_flat, flat_notes, heart_notes)
base_final = np.where(is_flat, flat_notes, base_notes)

olfactory_notes = [
    f"Notes: {f}" if flat else " | ".join(
        f"{lbl}: {txt}" for lbl, txt in [("Top", t), ("Heart", h), ("Base", b)] if txt
    )
    for flat, t, h, b, f in zip(is_flat, top_notes, heart_notes, base_notes, flat_notes)
]

# Moment (vectorisé)
total_dn = df_bal["day"] + df_bal["night"]
ratio_day = np.divide(df_bal["day"], total_dn, out=np.full(len(df_bal), 0.5), where=total_dn > 0)
best_moment = pd.Series(np.select(
    [ratio_day >= 0.60, ratio_day <= 0.40],
    ["Day", "Evening / Night"],
    default="Day & Night"
))

# Saisons standardisées (ordre chronologique fixe)
season_vals = df_bal[["spring", "summer", "autumn", "winter"]].to_numpy()
best_seasons = pd.Series([
    ", ".join(SEASON_ORDER[i] for i in sorted(np.argsort(-row)[:2]))
    for row in season_vals
])

# Budget (vectorisé)
b_low = df_bal["brand"].str.strip().str.lower()
pv = df_bal["price_value_avg"]
budget_tier = np.select(
    [b_low.isin(ACCESSIBLE_BRANDS), b_low.isin(LUXURY_NICHE_BRANDS), pv >= 4.2, pv <= 2.6],
    ["€ (< 60€)", "€€€ (> 150€)", "€ (< 60€)", "€€€ (> 150€)"],
    default="€€ (60-150€)"
)

desc_clean = df_bal["description"].str.replace(r"\s+", " ", regex=True).str.strip()
keywords_clean = df_bal["accords"].apply(lambda x: clean_pipe_list(x, 5))
occasion_clean = best_moment + " (" + best_seasons + ")"
olfactory_series = pd.Series(olfactory_notes)

final_df = pd.DataFrame({
    "PerfumeID": [f"P{i+1:03d}" for i in range(len(df_bal))],
    "Name": df_bal["name"].str.strip(),
    "Brand": df_bal["brand"].str.strip(),
    "Gender": df_bal["gender"],
    "Family": df_bal["Family"],
    "Budget": budget_tier,
    "TopNotes": top_final,
    "HeartNotes": heart_final,
    "BaseNotes": base_final,
    "OlfactoryNotes": olfactory_series,
    "Description": desc_clean,
    "Keywords": keywords_clean,
    "Occasion": occasion_clean,
    "BestMoment": best_moment,
    "BestSeasons": best_seasons,
    "Longevity": df_bal["longevity_avg"].round(2),
    "Sillage": df_bal["sillage_avg"].round(2),
    "Rating": df_bal["rating_avg"].round(2),
    "VoteCount": df_bal["vote_count"].astype(int),
    "SearchText": (
        "Family: " + df_bal["Family"] +
        ". Accords: " + keywords_clean +
        ". Occasion: " + occasion_clean +
        ". " + olfactory_series +
        ". Description: " + desc_clean.str.slice(0, 600)
    ),
})

final_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

# Check controle
print(f"Dataset sauvegardé : {final_df.shape} -> {OUTPUT_CSV}\n")
print(pd.crosstab(final_df["Family"], final_df["Gender"]))
print("\n", final_df["Budget"].value_counts())
print("\n", final_df[["PerfumeID", "Name", "TopNotes", "HeartNotes", "BaseNotes"]].head(3))

Using Colab cache for faster access to the 'fragrantica-perfumes' dataset.
Dataset sauvegardé : (200, 20) -> perfumes_200_clean.csv

Gender             female  male  unisex
Family                                 
Amber & Spicy           8     8       9
Aromatic                8     8       9
Citrus & Fresh          8     8       9
Floral                  8     8       9
Fruity                  8     8       9
Leather & Powdery       8     8       9
Sweet & Gourmand        8     8       9
Woody                   8     8       9

 Budget
€€ (60-150€)    111
€€€ (> 150€)     70
€ (< 60€)        19
Name: count, dtype: int64

   PerfumeID                Name  \
0      P001        Black Orchid   
1      P002  Coco Eau de Parfum   
2      P003           Coco Noir   

                                            TopNotes  \
0  Truffle, Gardenia, Black Currant, Ylang-Ylang,...   
1  Bulgarian Rose, Coriander, Peach, Jasmine, Man...   
2                       Grapefruit, Bergamot, Orange   

    

In [4]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)  # Évite que Description prenne tout l'écran

df = pd.read_csv("perfumes_200_clean.csv")
df.head(5)

,PerfumeID,Name,Brand,Gender,Family,Budget,TopNotes,HeartNotes,BaseNotes,OlfactoryNotes,Description,Keywords,Occasion,BestMoment,BestSeasons,Longevity,Sillage,Rating,VoteCount,SearchText
0,P001,Black Orchid,Tom Ford,female,Amber & Spicy,€€€ (> 150€),"Truffle, Gardenia, Black Currant, Ylang-Ylang,...","Orchid, Spices, Gardenia, Fruity Notes, Ylang-...","Mexican chocolate, Patchouli, Vanille, Incense...","Top: Truffle, Gardenia, Black Currant, Ylang-Y...",Black Orchid by Tom Ford is a Oriental Floral ...,"warm spicy, earthy, woody, sweet, amber","Evening / Night (Autumn, Winter)",Evening / Night,"Autumn, Winter",4.09,2.94,3.88,32626,"Family: Amber & Spicy. Accords: warm spicy, ea..."
1,P002,Coco Eau de Parfum,Chanel,female,Amber & Spicy,€€ (60-150€),"Bulgarian Rose, Coriander, Peach, Jasmine, Man...","Cloves, Rose, Mimosa, Orange Blossom, Clover","Amber, Sandalwood, Opoponax, Tonka Bean, Civet","Top: Bulgarian Rose, Coriander, Peach, Jasmine...",Coco Eau de Parfum by Chanel is a Oriental Spi...,"amber, warm spicy, sweet, powdery, woody","Evening / Night (Autumn, Winter)",Evening / Night,"Autumn, Winter",4.02,2.86,4.25,13088,"Family: Amber & Spicy. Accords: amber, warm sp..."
2,P003,Coco Noir,Chanel,female,Amber & Spicy,€€ (60-150€),"Grapefruit, Bergamot, Orange","Rose, Geranium, Jasmine, Narcissus, Peach","Patchouli, Sandalwood, Olibanum, Tonka Bean, V...","Top: Grapefruit, Bergamot, Orange | Heart: Ros...",Coco Noir by Chanel is a Oriental Woody fragra...,"warm spicy, woody, citrus, fresh spicy, amber","Evening / Night (Autumn, Winter)",Evening / Night,"Autumn, Winter",3.50,2.44,4.13,12063,"Family: Amber & Spicy. Accords: warm spicy, wo..."
3,P004,Alien Essence Absolue,Mugler,female,Amber & Spicy,€€ (60-150€),Jasmine,"Cashmere Wood, Orris Root","Vanilla Pod, White Amber, Myrrh, Incense","Top: Jasmine | Heart: Cashmere Wood, Orris Roo...",Alien Essence Absolue by Mugler is a Oriental ...,"amber, vanilla, white floral, balsamic, powdery","Day & Night (Autumn, Winter)",Day & Night,"Autumn, Winter",4.12,2.94,4.35,6315,"Family: Amber & Spicy. Accords: amber, vanilla..."
4,P005,Portrait of a Lady,Frederic Malle Editions de Parfums,female,Amber & Spicy,€€€ (> 150€),"Rose, Clove, Raspberry, Black Currant, Cinnamon","Turkish Rose, Patchouli, Incense, Sandalwood, ...","Incense, Sandalwood, Musk, Benzoin, Amber","Top: Rose, Clove, Raspberry, Black Currant, Ci...",Portrait of a Lady by Frederic Malle Editions ...,"warm spicy, amber, rose, woody, patchouli","Evening / Night (Autumn, Winter)",Evening / Night,"Autumn, Winter",4.09,2.93,4.06,11941,"Family: Amber & Spicy. Accords: warm spicy, am..."
